In [1]:
!pip install unsloth
!pip install -q transformers accelerate datasets bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 4.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 372.2/372.2 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 7.7 MB/s eta 0:00:00:00:0100:01m
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.6/288.6 kB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 122.9/122.9 MB 5.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 654.9 kB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 MB 5.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 1.2 MB/s eta 0:00:00:00:01

In [ ]:
# Install required packages (run this at the top of your Colab session)
!pip install unsloth transformers accelerate datasets bitsandbytes

from unsloth import FastLanguageModel
from datasets import load_dataset
from transformers import TrainingArguments, Trainer, BitsAndBytesConfig
import torch

# Set model and sequence length
BASE_MODEL = "Qwen/Qwen2.5-1.8B-Instruct"
SEQ_LEN = 2048

# Load model with 4-bit quantization and CPU offload for Colab compatibility
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    llm_int8_enable_fp32_cpu_offload=True,
)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=SEQ_LEN,
    dtype="bfloat16" if torch.cuda.is_bf16_supported() else "float16",
    quantization_config=bnb_config,
)

# Load your prepared data file from Google Drive
train = load_dataset("json", data_files="/content/drive/MyDrive/sample_track_a_prepared.jsonl")['train']

def format_chat(x):
    return {"text": x["prompt"] + str(x["answer"])}

train = train.map(format_chat)

def tokenize_function(example):
    text = example["text"]
    if isinstance(text, list):
        text = " ".join([str(t) for t in text])
    else:
        text = str(text)
    result = tokenizer(
        text,
        truncation=True,
        max_length=SEQ_LEN,
        padding="max_length",
    )
    result["labels"] = result["input_ids"].copy()
    return result

tokenized_train = train.map(tokenize_function, batched=False)

# Prepare LoRA/PEFT model
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    lora_dropout=0.0,
)

# Training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/models/qwen2.5-1.8b-trackA",
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    save_total_limit=1,
    logging_steps=10,
    save_steps=100,
    report_to="none",
    remove_unused_columns=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    tokenizer=tokenizer,
)

trainer.train()
model.save_pretrained("/content/drive/MyDrive/models/qwen2.5-1.8b-trackA")

NotImplementedError: Unsloth cannot find any torch accelerator? You need a GPU.

In [3]:
# Check for GPU availability in PyTorch
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
else:
    print('No GPU detected. Unsloth requires a supported GPU (NVIDIA, AMD, or Intel).')

CUDA available: False
No GPU detected. Unsloth requires a supported GPU (NVIDIA, AMD, or Intel).
